# 第8章 車を「動かして」未来を見る (行動条件付き生成 = Forward Dynamics)

第7章は「どうなってほしいか」を文章で伝えて動画を作りました。この章では、文章の代わりに**行動 (action) の数値**を渡します。
「この画像の場所から、車がこう動いたら、カメラにはどう映るか」を予測させるので、**順動力学 (forward dynamics)** と呼びます。
ゲームのように操作を変えると未来が変わる、世界モデルらしい使い方です。

1. 行動の数値の形を知る (公式の軌跡を上から見た図にする)
2. 公式の画像と軌跡で 6 秒先まで予測する
3. 自分で軌跡を作る (直進・左折・右折・停止) → 同じ画像から 4 通りの未来を並べる
4. 自分のドライブレコーダーの画像で試す
5. 車ではなくカメラを動かす (camera pose)

設定は公式の推奨どおりです: 60 ステップ (10fps で 6 秒) の行動、480p、`num_inference_steps=30`、`guidance_scale=1.0`、`flow_shift=10.0`。
目安: 1 本あたり A100 で 1〜2 分、L4 ではその数倍。

## 準備

**ランタイム**: 「ランタイム → ランタイムのタイプを変更」で **L4** か **A100** の GPU を選んでください (T4 は対象外)。
A100 (40GB 以上) ではモデル全体を GPU に載せ、L4 (24GB) では使う部品だけを GPU に載せる設定 (CPU オフロード) に自動で切り替えます。

**安全フィルタ**: 生成には NVIDIA の安全フィルタ (`cosmos_guardrail`) を使います。使うには Hugging Face で
[nvidia/Cosmos-1.0-Guardrail](https://huggingface.co/nvidia/Cosmos-1.0-Guardrail) の利用申請をして承認を受け、
Colab の左の鍵のマーク (シークレット) に `HF_TOKEN` を登録しておいてください。承認を待たずに試すときは、下の `GUARDRAIL` の印を外します。

In [ ]:
# @title GPU を確かめる
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
if DEVICE == "cuda":
    GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(torch.cuda.get_device_name(0), f"{GPU_GB:.0f} GB")
    assert torch.cuda.get_device_capability()[0] >= 8, "Ampere 以降の GPU (L4 / A100 など) を選んでください。T4 では動きません。"
else:
    GPU_GB = 0
    print(f"CUDA の GPU が見つかりません ({DEVICE} で動かします。とても遅くなります)")

In [ ]:
# @title インストール (数分)
# Cosmos3OmniPipeline は Diffusers の main ブランチに入っている
!pip install -q -U "transformers>=5.18" accelerate av imageio imageio-ffmpeg huggingface_hub
!pip install -q "diffusers @ git+https://github.com/huggingface/diffusers.git" cosmos_guardrail

In [ ]:
# @title 安全フィルタの設定とサンプル素材を取ってくる関数
import os
import urllib.request
from pathlib import Path

GUARDRAIL = True  # @param {type:"boolean"}
try:  # Colab のシークレットに HF_TOKEN があれば使う
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass

ASSETS = Path(os.environ.get("COSMOS_ASSETS", "assets"))
ASSETS.mkdir(parents=True, exist_ok=True)
FW = "https://media.githubusercontent.com/media/NVIDIA/cosmos-framework/main/.agents/skills/cookbooks/cosmos3"
HF = "https://huggingface.co/nvidia/Cosmos3-Edge/resolve/main/assets"


def fetch(url):
    """URL の素材を assets/ に落として、そのパスを返す (2 回目からは落とさない)"""
    path = ASSETS / url.rsplit("/", 1)[-1]
    if not path.exists():
        urllib.request.urlretrieve(url, path)
    return str(path)

In [ ]:
# @title 生成モデル (Generator) を読み込む関数
import gc
import json

from diffusers import Cosmos3OmniPipeline
from diffusers.schedulers.scheduling_unipc_multistep import UniPCMultistepScheduler
from diffusers.utils import export_to_video
from IPython.display import Image as IPImage, Video, display
from PIL import Image, ImageOps


def load_image(path, max_side=10_000):
    """画像を RGB で読む (max_side より大きければ縮める)"""
    img = path if isinstance(path, Image.Image) else ImageOps.exif_transpose(Image.open(path))
    img = img.convert("RGB")
    img.thumbnail((max_side, max_side))
    return img


def load_pipe(flow_shift):
    """Cosmos3OmniPipeline を読み込む (2 回目からは読み込み済みのものを使う)。
    flow_shift は公式の推奨値: 画像→動画は 12.0、行動 (action) を使う生成は 10.0"""
    global pipe
    if "pipe" not in globals():
        pipe = Cosmos3OmniPipeline.from_pretrained(
            "nvidia/Cosmos3-Edge", torch_dtype=torch.bfloat16, enable_safety_checker=GUARDRAIL)
        if DEVICE == "cuda" and GPU_GB < 40:
            pipe.enable_model_cpu_offload()  # L4 など: 使う部品だけ GPU に載せる
        else:
            pipe.to(DEVICE)
    pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config, flow_shift=flow_shift, use_karras_sigmas=False)
    return pipe


def free_gpu():
    """読み込んだモデルを GPU から降ろす"""
    for name in ("pipe", "model", "processor"):
        globals().pop(name, None)
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()


def show_video(path, width=640):
    display(Video(path, embed=True, width=width))

In [ ]:
# @title 行動 (action) を作る・描く関数
import numpy as np
import matplotlib.pyplot as plt
from diffusers import CosmosActionCondition

RAW = "https://raw.githubusercontent.com/NVIDIA/cosmos-framework/main/.agents/skills/cookbooks/cosmos3"
AV_PROMPT = "You are an autonomous vehicle planning system."
# AV の行動は「1 コマ前のカメラから見た、次のコマのカメラの位置と向き」を 9 個の数で表す。
#   [x, y, z, r1..r6]  x: 右, y: 下, z: 前 (OpenCV のカメラ座標)。r1..r6: 回転行列の 1 列目と 2 列目 (rot6d)
# 移動量はメートルを 1.35 倍した値 (公式の可視化コードと同じ換算)
AV_SCALE = 1.35


def rot_y(theta):
    """カメラの下向きの軸 (y) まわりの回転。theta が負だと左を向く"""
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, 0, s], [0, 1, 0], [-s, 0, c]])


def from_rot6d(r):
    a = np.asarray(r[:3], float)
    b = np.asarray(r[3:6], float)
    c0 = a / np.linalg.norm(a)
    c1 = b - c0 * (c0 @ b)
    c1 /= np.linalg.norm(c1)
    return np.stack([c0, c1, np.cross(c0, c1)], axis=1)


def make_av_actions(speed_kmh=30.0, left_deg_per_s=0.0, accel=0.0, seconds=6.0, fps=10):
    """速さ (km/h)・左へ曲がる速さ (度/秒、右は負)・加速度 (m/s^2、減速は負) から AV の行動 (T, 9) を作る。
    left_deg_per_s には「時刻 t (秒) → 度/秒」の関数も渡せる (車線変更などに)"""
    rows, dt = [], 1 / fps
    for i in range(int(round(seconds * fps))):
        t = i * dt
        v = max(speed_kmh / 3.6 + accel * t, 0.0)
        d = v * dt
        yaw = left_deg_per_s(t) if callable(left_deg_per_s) else left_deg_per_s
        th = -np.radians(yaw) * dt if d > 0 else 0.0
        move = np.array([np.sin(th / 2) * d, 0.0, np.cos(th / 2) * d]) * AV_SCALE
        R = rot_y(th)
        rows.append(np.concatenate([move, R[:, 0], R[:, 1]]))
    return np.array(rows, np.float32)


def to_path(actions, scale=AV_SCALE):
    """行動 (T, 9) をつなげて、上から見た道筋 (T+1, 3) にする。x: 右, z: 前 (メートル)"""
    R, p, pts = np.eye(3), np.zeros(3), [np.zeros(3)]
    for a in np.asarray(actions, float):
        p = p + R @ (a[:3] / scale)
        R = R @ from_rot6d(a[3:9])
        pts.append(p.copy())
    return np.array(pts)


def plot_paths(paths, title="top-down view", fps=10):
    """{名前: 行動 (T, 9)} を上から見た図に描く。点は 1 秒ごと"""
    fig, ax = plt.subplots(figsize=(5, 5))
    for i, (name, actions) in enumerate(paths.items()):
        p = to_path(actions)
        ax.plot(p[:, 0], p[:, 2], lw=2, label=name)
        ax.scatter(p[::fps, 0], p[::fps, 2], s=12)
    ax.scatter([0], [0], c="k", marker="^", s=80, zorder=5)
    ax.set_xlabel("x: right (m)")
    ax.set_ylabel("z: forward (m)")
    ax.set_aspect("equal", adjustable="datalim")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
    ax.set_title(title)
    plt.show()


def tile_videos(videos, labels, out, fps, cols=2):
    """同じ長さの動画 (PIL 画像のリスト) を並べて 1 本にする"""
    from PIL import ImageDraw, ImageFont
    n = min(len(v) for v in videos)
    w, h = videos[0][0].size
    rows = (len(videos) + cols - 1) // cols
    font = ImageFont.load_default(size=max(16, w // 25))
    frames = []
    for t in range(n):
        canvas = Image.new("RGB", (w * cols, h * rows))
        for k, (v, label) in enumerate(zip(videos, labels)):
            tile = v[t].convert("RGB").resize((w, h))
            ImageDraw.Draw(tile).text((10, 8), label, fill="white", font=font, stroke_width=3, stroke_fill="black")
            canvas.paste(tile, ((k % cols) * w, (k // cols) * h))
        frames.append(canvas)
    export_to_video(frames, out, fps=fps, macro_block_size=1)
    return out

## 1. 行動の数値の形を知る

AV (自動運転) の行動は、1 コマごとに 9 個の数です。

| 位置 | 意味 |
| --- | --- |
| 0〜2 | 1 コマ前のカメラから見た移動 (x: 右、y: 下、z: 前)。メートル × 1.35 |
| 3〜8 | 1 コマ前のカメラから見た向きの変化。回転行列の 1 列目と 2 列目 (rot6d) |

60 コマ分つなげると、上から見た車の道筋になります。公式の「直進」「左折」「右折」の 3 本を描いてみます。

In [ ]:
official = {name: np.array(json.loads(Path(fetch(f"{RAW}/generator/action/assets/actions/av_traj_{name}.json")).read_text()), np.float32)
            for name in ("forward", "left", "right")}
print("形:", official["left"].shape, "(コマ数, 9)")
print("左折の最初の 1 コマ:", np.round(official["left"][0], 3))
plot_paths(official, "official AV trajectories (6 s)")

## 2. 公式の画像と軌跡で 6 秒先まで予測する

行動を使うときは、プロンプトは JSON ではなく短い決まり文句 (`You are an autonomous vehicle planning system.`) にします (公式の注意: アップサンプリングしない)。
画像・行動・設定は `CosmosActionCondition` にまとめて渡します。動画の長さは行動の数 + 1 コマ (最初のコマ) で決まります。

In [ ]:
start = load_image(fetch(f"{FW}/generator/action/assets/images/av_0.jpg"))
display(start.resize((416, 240)))


def drive(image, actions, seed=0):
    """画像と AV の行動 (T, 9) から、T+1 コマの動画 (PIL 画像のリスト) を作る"""
    pipe = load_pipe(flow_shift=10.0)
    result = pipe(
        prompt=AV_PROMPT,
        action=CosmosActionCondition(
            mode="forward_dynamics",
            chunk_size=len(actions),
            domain_name="av",
            resolution_tier=480,
            raw_actions=torch.as_tensor(actions, dtype=torch.float32),
            image=image,
            view_point="ego_view",
        ),
        fps=10,
        num_inference_steps=30,
        guidance_scale=1.0,
        use_system_prompt=False,
        generator=torch.Generator(device=DEVICE).manual_seed(seed),
    )
    return result.video


frames = drive(start, official["left"])
export_to_video(frames, "ch08_official_left.mp4", fps=10, macro_block_size=1)
show_video("ch08_official_left.mp4")

## 3. 自分で軌跡を作る

`make_av_actions` は、速さ・曲がる速さ・加速度から 60 コマ分の行動を作ります。
同じ最初のコマから 4 通りの操作をして、未来がどう変わるかを並べて比べます。4 本作るので時間がかかります。

In [ ]:
maneuvers = {
    "straight 40km/h": make_av_actions(speed_kmh=40),
    "turn left": make_av_actions(speed_kmh=20, left_deg_per_s=15),
    "turn right": make_av_actions(speed_kmh=20, left_deg_per_s=-15),
    "brake to stop": make_av_actions(speed_kmh=30, accel=-3.0),
}
plot_paths(maneuvers, "my trajectories")

In [ ]:
videos = {}
for name, actions in maneuvers.items():
    videos[name] = drive(start, actions)
    print("done:", name)
tile_videos(list(videos.values()), list(videos), "ch08_four_futures.mp4", fps=10)
show_video("ch08_four_futures.mp4", 832)

「左折」と「右折」で景色が逆に流れ、「停止」では途中から景色が止まれば、行動どおりに未来を描けています。
ただし、交差点でない場所で曲がる指示を出すと、道のない所に進んだり景色が崩れたりします。**世界モデルは見たことのない状況ほど苦手**です。

## 4. 自分のドライブレコーダーの画像で試す

画像は 16:9 (832×480) に合わせて中央を切り抜きます。日付や速度の表示が下に入っているときは `crop_bottom` で切り落とすときれいになります。

In [ ]:
# @title 画像と操作を選ぶ (colab-only)
from google.colab import files
from PIL import ImageOps

maneuver = "straight 40km/h"  # @param ["straight 40km/h", "turn left", "turn right", "brake to stop"]
crop_bottom = 0.0  # @param {type:"slider", min:0, max:0.3, step:0.01}

up = files.upload()
img = load_image(next(iter(up)), max_side=10_000)
if crop_bottom > 0:
    img = img.crop((0, 0, img.width, round(img.height * (1 - crop_bottom))))
img = ImageOps.fit(img, (832, 480), Image.BICUBIC)
display(img.resize((416, 240)))
frames = drive(img, maneuvers[maneuver])
export_to_video(frames, "ch08_my_drive.mp4", fps=10, macro_block_size=1)
show_video("ch08_my_drive.mp4")

## 5. 車ではなくカメラを動かす

`domain_name="camera_pose"` にすると、同じ 9 個の数の形で「カメラの動き」を渡せます。風景写真の中をカメラが飛んでいくような動画になります。
こちらは 30fps で、プロンプトは場面を説明する短い文章です。公式の灯台の例と、その動きを左右反転した例を並べます。

In [ ]:
cam = np.array(json.loads(Path(fetch(f"{RAW}/generator/action/assets/actions/camera_action.json")).read_text()), np.float32)
S = np.diag([-1.0, 1.0, 1.0])  # 左右反転: 移動の x の符号を変え、回転を S R S にする
mirrored = cam.copy()
for i, a in enumerate(cam):
    R = S @ from_rot6d(a[3:9]) @ S
    mirrored[i] = np.concatenate([S @ a[:3], R[:, 0], R[:, 1]])

lighthouse = load_image(fetch(f"{FW}/generator/action/assets/images/lighthouse_720.png"), max_side=10_000)
prompt = Path(fetch(f"{RAW}/generator/action/assets/prompts/lighthouse.txt")).read_text().strip()
print(prompt)

pipe = load_pipe(flow_shift=10.0)
cams = []
for actions in (cam, mirrored):
    result = pipe(
        prompt=prompt,
        action=CosmosActionCondition(mode="forward_dynamics", chunk_size=len(actions), domain_name="camera_pose",
                                     resolution_tier=480, raw_actions=torch.as_tensor(actions), image=lighthouse,
                                     view_point="ego_view"),
        fps=30, num_inference_steps=30, guidance_scale=1.0, use_system_prompt=False,
        generator=torch.Generator(device=DEVICE).manual_seed(0),
    )
    cams.append(result.video)
tile_videos(cams, ["official", "mirrored"], "ch08_camera.mp4", fps=30)
show_video("ch08_camera.mp4", 832)

## うまくいかないとき

- **行動どおりに動かない**: 速すぎる・急すぎる操作は学習データにないので無視されがちです。まずは 20〜50 km/h、曲がる速さ 20 度/秒以内から試してください
- **景色が崩れる**: seed を変えて作り直すか、より道路が広く写った画像を使います
- **時間がかかる**: 行動を 30 コマ (`seconds=3`) に減らすと速くなります
- 作られる動画はもっともらしい予測で、本当にその道を走ったときの映像ではありません。第9章では、この予測動画から逆に軌跡を当て直して、どれくらい行動どおりかを確かめます